# ANTARES Cumulative Nightly History

This notebook builds the Drive-backed nightly history store without replacing the normal `alerts_time_comparison.ipynb` workflow.

In [1]:
%cd /content
!rm -rf ANTARES_Analysis
!git clone https://github.com/darim1151/ANTARES_Analysis.git
%cd ANTARES_Analysis
!git pull --ff-only
!ls src

/content
Cloning into 'ANTARES_Analysis'...
remote: Enumerating objects: 64, done.
remote: Counting objects: 100% (64/64), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 64 (delta 13), reused 40 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (64/64), 5.89 MiB | 8.74 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/ANTARES_Analysis
Already up to date.
cache.py	  config.py   history.py   lightcurves.py  summary.py
chunked_query.py  figures.py  __init__.py  query.py	   validation.py


In [2]:
!pip install --quiet antares-client elasticsearch-dsl astropy matplotlib pandas numpy pyarrow

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.8/952.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.0/191.0 kB 15.5 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/ANTARES_Analysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config, history

DATA_ROOT = config.HISTORY_DATA_ROOT
MJD_HISTORY_START = config.MJD2_MIN
MJD_HISTORY_CUTOFF = config.MJD2_MAX

print('Ready.')
config.print_config_summary()
print('\nHistory store:', DATA_ROOT)

Ready.
Configuration
  Last Night: MJD 61102.0 - 61103.0  (1.0 days)  [OK]
  LSST History: MJD 60200.0 - 61102.0  (902.0 days)  [OK]
  Samples per range : 5000
  Tag filter        : none (all alerts)
  Random seed       : 42
  Chunked ingest    : ON
  Chunk start size  : 1 day(s)
  Chunk min size    : 30 sec
  Chunk split at    : 9,500/10,000 loci
  History backfill  : OFF
  History data root : /content/drive/MyDrive/ANTARES_Analysis
  History target    : 100,000 loci/night
  History LC fetch  : ON

  Ranges are NON-overlapping  (MJD2_MAX=61102.0, MJD1_MIN=61102.0)

History store: /content/drive/MyDrive/ANTARES_Analysis


## 1. Tiny Smoke Backfill

This creates one nightly partition with a small target and skips lightcurves. Use it first to verify Drive paths, manifests, parquet writing, and resume behavior.

In [5]:
smoke_summary = history.backfill_history(
    data_root=DATA_ROOT,
    mjd_start=MJD_HISTORY_START,
    mjd_stop=MJD_HISTORY_START + 1,
    target_loci=1000,
    max_nights=1,
    fetch_lightcurves=False,
    resume=True,
)
display(smoke_summary.tail())

Backfill 2023/9/13  MJD [60200.000000, 60201.000000]
  Chunked query 'History 2023/9/13'  MJD [60200.000000, 60201.000000]
    ES limit=10,000, split at >= 9,500, minimum chunk=30s
       1. 60200.000000-60201.000000   86400.0s  10,000 loci  split  (live; 2 queued)
       2. 60200.000000-60200.500000   43200.0s  10,000 loci  split  (live; 3 queued)
       3. 60200.000000-60200.250000   21600.0s   4,653 loci  accepted  (live; 2 queued)
    target reached (4,653/1,000); stopping with 2 unqueried chunks.
  History 2023/9/13: 1,000 unique loci from 1 accepted chunks  (2 splits, 268.1s)
  Lightcurve fetch disabled for this run.


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,duplicate_locus_count,coordinate_pass,overlap_count,alert_locus_link_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2023-09-13,2023/9/13,60200.0,60201.0,None,1000,1000,0,1,2,...,0,True,0,True,268.45,2026-04-29T20:00:16+00:00,2026-04-29T20:04:44+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...


In [6]:
resume_summary = history.backfill_history(
    data_root=DATA_ROOT,
    mjd_start=MJD_HISTORY_START,
    mjd_stop=MJD_HISTORY_START + 1,
    target_loci=1000,
    max_nights=1,
    fetch_lightcurves=False,
    resume=True,
)
display(resume_summary.tail())

Backfill 2023/9/13  MJD [60200.000000, 60201.000000]
  Resume: found 2023-09-13; loading existing nightly partition.


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,duplicate_locus_count,coordinate_pass,overlap_count,alert_locus_link_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2023-09-13,2023/9/13,60200.0,60201.0,None,1000,1000,0,1,2,...,0,True,0,True,268.45,2026-04-29T20:00:16+00:00,2026-04-29T20:04:44+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...


## 2. Three-Night Test

Turn this on after the smoke test. It checks multi-night folder layout and cumulative-index behavior.

In [13]:
RUN_THREE_NIGHT_TEST = True

if RUN_THREE_NIGHT_TEST:
    three_night_summary = history.backfill_history(
        data_root=DATA_ROOT,
        mjd_start=MJD_HISTORY_START,
        mjd_stop=MJD_HISTORY_START + 3,
        target_loci=5000,
        max_nights=3,
        fetch_lightcurves=False,
        resume=True,
    )
    display(three_night_summary.tail(10))

Backfill 2023/9/13  MJD [60200.000000, 60201.000000]
  Resume: found 2023-09-13; loading existing nightly partition.
Backfill 2023/9/14  MJD [60201.000000, 60202.000000]
  Chunked query 'History 2023/9/14'  MJD [60201.000000, 60202.000000]
    ES limit=10,000, split at >= 9,500, minimum chunk=30s
       1. 60201.000000-60202.000000   86400.0s  10,000 loci  split  (live; 2 queued)
       2. 60201.000000-60201.500000   43200.0s  10,000 loci  split  (live; 3 queued)
       3. 60201.000000-60201.250000   21600.0s   6,627 loci  accepted  (live; 2 queued)
    target reached (6,627/5,000); stopping with 2 unqueried chunks.
  History 2023/9/14: 5,000 unique loci from 1 accepted chunks  (2 splits, 280.3s)
  Lightcurve fetch disabled for this run.
Backfill 2023/9/15  MJD [60202.000000, 60203.000000]
  Chunked query 'History 2023/9/15'  MJD [60202.000000, 60203.000000]
    ES limit=10,000, split at >= 9,500, minimum chunk=30s
       1. 60202.000000-60203.000000   86400.0s  10,000 loci  split  (li

,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,duplicate_locus_count,coordinate_pass,overlap_count,alert_locus_link_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2023-09-13,2023/9/13,60200.0,60201.0,None,1000,1000,0,1,2,...,0,True,0,True,268.45,2026-04-29T20:00:16+00:00,2026-04-29T20:04:44+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
1,2023-09-14,2023/9/14,60201.0,60202.0,None,5000,5000,0,1,2,...,0,True,0,True,280.59,2026-04-29T20:11:22+00:00,2026-04-29T20:16:03+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
2,2023-09-15,2023/9/15,60202.0,60203.0,None,5000,5000,0,3,5,...,0,True,0,True,577.83,2026-04-29T20:16:03+00:00,2026-04-29T20:25:41+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...


## 3. Full-Scale Runs

Run one full night first. Only then turn on the full historical backfill.

In [15]:
RUN_ONE_FULL_NIGHT = True
RUN_FULL_BACKFILL = True

if RUN_ONE_FULL_NIGHT:
    one_full_summary = history.backfill_history(
        data_root=DATA_ROOT,
        mjd_start=MJD_HISTORY_START,
        mjd_stop=MJD_HISTORY_START + 1,
        target_loci=config.HISTORY_TARGET_LOCI,
        max_nights=1,
        fetch_lightcurves=config.HISTORY_FETCH_ALL_LIGHTCURVES,
        resume=True,
    )
    display(one_full_summary.tail())

if RUN_FULL_BACKFILL:
    full_summary = history.backfill_history(
        data_root=DATA_ROOT,
        mjd_start=MJD_HISTORY_START,
        mjd_stop=MJD_HISTORY_START+5,
        target_loci=config.HISTORY_TARGET_LOCI,
        max_nights=5,
        fetch_lightcurves=config.HISTORY_FETCH_ALL_LIGHTCURVES,
        resume=True,
    )
    display(full_summary.tail(20))

Backfill 2023/9/13  MJD [60200.000000, 60201.000000]
  Resume: found 2023-09-13; loading existing nightly partition.


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,duplicate_locus_count,coordinate_pass,overlap_count,alert_locus_link_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
2,2023-09-15,2023/9/15,60202.0,60203.0,None,5000,5000,0,3,5,...,0.0,True,0.0,True,577.83,2026-04-29T20:16:03+00:00,2026-04-29T20:25:41+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
3,2023-09-16,2023/9/16,60203.0,60204.0,None,100000,43917,1062359,10,9,...,0.0,True,0.0,True,2033.16,2026-04-29T22:12:23+00:00,2026-04-29T22:46:16+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
4,2023-09-17,2023/9/17,60204.0,60205.0,None,100000,41996,1130681,6,5,...,0.0,True,0.0,True,1572.41,2026-04-29T22:46:17+00:00,2026-04-29T23:12:29+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
5,2023-09-18,2023/9/18,60205.0,60206.0,None,100000,53847,1237449,9,8,...,0.0,True,0.0,True,1991.51,2026-04-29T23:12:31+00:00,2026-04-29T23:45:43+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
6,2023-09-19,2023/9/19,60206.0,60207.0,None,100000,0,0,0,0,...,NaN,None,NaN,None,68.57,2026-04-29T23:45:46+00:00,2026-04-29T23:46:54+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...


Backfill 2023/9/13  MJD [60200.000000, 60201.000000]
  Resume: found 2023-09-13; loading existing nightly partition.
Backfill 2023/9/14  MJD [60201.000000, 60202.000000]
  Resume: found 2023-09-14; loading existing nightly partition.
Backfill 2023/9/15  MJD [60202.000000, 60203.000000]
  Resume: found 2023-09-15; loading existing nightly partition.
Backfill 2023/9/16  MJD [60203.000000, 60204.000000]
  Resume: found 2023-09-16; loading existing nightly partition.
Backfill 2023/9/17  MJD [60204.000000, 60205.000000]
  Resume: found 2023-09-17; loading existing nightly partition.


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,duplicate_locus_count,coordinate_pass,overlap_count,alert_locus_link_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2023-09-13,2023/9/13,60200.0,60201.0,None,1000,1000,0,1,2,...,0.0,True,0.0,True,268.45,2026-04-29T20:00:16+00:00,2026-04-29T20:04:44+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
1,2023-09-14,2023/9/14,60201.0,60202.0,None,5000,5000,0,1,2,...,0.0,True,0.0,True,280.59,2026-04-29T20:11:22+00:00,2026-04-29T20:16:03+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
2,2023-09-15,2023/9/15,60202.0,60203.0,None,5000,5000,0,3,5,...,0.0,True,0.0,True,577.83,2026-04-29T20:16:03+00:00,2026-04-29T20:25:41+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
3,2023-09-16,2023/9/16,60203.0,60204.0,None,100000,43917,1062359,10,9,...,0.0,True,0.0,True,2033.16,2026-04-29T22:12:23+00:00,2026-04-29T22:46:16+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
4,2023-09-17,2023/9/17,60204.0,60205.0,None,100000,41996,1130681,6,5,...,0.0,True,0.0,True,1572.41,2026-04-29T22:46:17+00:00,2026-04-29T23:12:29+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
5,2023-09-18,2023/9/18,60205.0,60206.0,None,100000,53847,1237449,9,8,...,0.0,True,0.0,True,1991.51,2026-04-29T23:12:31+00:00,2026-04-29T23:45:43+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
6,2023-09-19,2023/9/19,60206.0,60207.0,None,100000,0,0,0,0,...,NaN,None,NaN,None,68.57,2026-04-29T23:45:46+00:00,2026-04-29T23:46:54+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...


## 4. Nightly Update

This ingests the newest night, compares it against prior cumulative history, and appends it only after validation passes.

In [16]:
RUN_NIGHTLY_UPDATE = True

if RUN_NIGHTLY_UPDATE:
    nightly_result = history.run_nightly_update(
        data_root=DATA_ROOT,
        mjd_min=config.MJD1_MIN,
        mjd_max=config.MJD1_MAX,
        target_loci=config.HISTORY_TARGET_LOCI,
        fetch_lightcurves=config.HISTORY_FETCH_ALL_LIGHTCURVES,
        resume=True,
    )
    print(nightly_result['comparison'])
    print(nightly_result['manifest']['status'])

  Chunked query 'Last Night 2026/3/3'  MJD [61102.000000, 61103.000000]
    ES limit=10,000, split at >= 9,500, minimum chunk=30s
       1. 61102.000000-61103.000000   86400.0s  10,000 loci  split  (live; 2 queued)
       2. 61102.000000-61102.500000   43200.0s  10,000 loci  split  (live; 3 queued)
       3. 61102.000000-61102.250000   21600.0s  10,000 loci  split  (live; 4 queued)
       4. 61102.000000-61102.125000   10800.0s     678 loci  accepted  (live; 3 queued)
       5. 61102.125000-61102.250000   10800.0s  10,000 loci  split  (live; 4 queued)
       6. 61102.125000-61102.187500    5400.0s   7,373 loci  accepted  (live; 3 queued)
       7. 61102.187500-61102.250000    5400.0s   9,708 loci  split  (live; 4 queued)
       8. 61102.187500-61102.218750    2700.0s   6,389 loci  accepted  (live; 3 queued)
       9. 61102.218750-61102.250000    2700.0s   3,319 loci  accepted  (live; 2 queued)
      10. 61102.250000-61102.500000   21600.0s   6,469 loci  accepted  (live; 1 queued)
     

## 5. Inspect Stored Data

In [17]:
loci_index, nightly_summary = history.update_cumulative_indexes(DATA_ROOT)
print(f'Cumulative index rows: {len(loci_index):,}')
display(nightly_summary.tail(20))

Cumulative index rows: 174,220


,date_utc,display_date,mjd_min,mjd_max,query_tag,target_loci,actual_loci,alert_rows,chunk_count,split_count,...,duplicate_locus_count,coordinate_pass,overlap_count,alert_locus_link_pass,runtime_seconds,started_at_utc,finished_at_utc,loci_path,alerts_path,manifest_path
0,2023-09-13,2023/9/13,60200.0,60201.0,None,1000,1000,0,1,2,...,0.0,True,0.0,True,268.45,2026-04-29T20:00:16+00:00,2026-04-29T20:04:44+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
1,2023-09-14,2023/9/14,60201.0,60202.0,None,5000,5000,0,1,2,...,0.0,True,0.0,True,280.59,2026-04-29T20:11:22+00:00,2026-04-29T20:16:03+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
2,2023-09-15,2023/9/15,60202.0,60203.0,None,5000,5000,0,3,5,...,0.0,True,0.0,True,577.83,2026-04-29T20:16:03+00:00,2026-04-29T20:25:41+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
3,2023-09-16,2023/9/16,60203.0,60204.0,None,100000,43917,1062359,10,9,...,0.0,True,0.0,True,2033.16,2026-04-29T22:12:23+00:00,2026-04-29T22:46:16+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
4,2023-09-17,2023/9/17,60204.0,60205.0,None,100000,41996,1130681,6,5,...,0.0,True,0.0,True,1572.41,2026-04-29T22:46:17+00:00,2026-04-29T23:12:29+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
5,2023-09-18,2023/9/18,60205.0,60206.0,None,100000,53847,1237449,9,8,...,0.0,True,0.0,True,1991.51,2026-04-29T23:12:31+00:00,2026-04-29T23:45:43+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
6,2023-09-19,2023/9/19,60206.0,60207.0,None,100000,0,0,0,0,...,NaN,None,NaN,None,68.57,2026-04-29T23:45:46+00:00,2026-04-29T23:46:54+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
7,2026-03-03,2026/3/3,61102.0,61103.0,None,100000,23460,3933256,6,5,...,0.0,True,0.0,True,1532.27,2026-04-29T23:47:22+00:00,2026-04-30T00:12:55+00:00,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...,/content/drive/MyDrive/ANTARES_Analysis/data/n...
